#Are High-Paid Running Backs Worth It? A NFL Cap Analysis (2011 - 2025)

#Introduction
This project analyzes whether teams that invest heavily in running backs achieve playoff success. With salary cap data from Spotrac and team data from Pro Football Reference, I chose to record the top 5 highest-paid running backs by cap hit and contract average from each season (2011 to 2025) and analyzed how those teams finished. The goal of this project is to determine if investing siginificant money and cap space for a premium running back is worth it.


In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

#Connect to the database
conn = sqlite3.connect('database/rb_analysis.db')

#Reusable formatting function
def format_df(df):
    df = df.copy()
    if 'contract_average' in df.columns:
        df['contract_average'] = df['contract_average'].apply(lambda x: '${:,.0f}'.format(float(x)) if x != 'DNQ' else x)
    if 'cap_hit' in df.columns:
        df['cap_hit'] = df['cap_hit'].apply(lambda x: '${:,.0f}'.format(float(x)) if x != 'DNQ' else x)
    if 'cap_percentage' in df.columns:
        df['cap_percentage'] = df['cap_percentage'].apply(lambda x: '{}%'.format(x) if x != 'DNQ' and not str(x).endswith('%') else x)
    return df

## Data Overview
The dataset consists of two tables:
- **rb_top_5** — top paid running backs each season by contract average and/or cap hit
- **team_data** — season outcomes for those teams including wins, losses, and playoff result

Data spans from **2011 to 2025**: Chosen because the 2011 CBA introduced the current rookie wage scale and standardized contract structures, making salary comparisons consistent across all recorded seasons.

The 2011 CBA is a critical starting point for this project because prior to 2011, rookie contracts were largley unregulated. This means that teams could spend a huge amount on first round picks at any position including running backs. But with the 2011 CBA it introduced a rookie wage scale that greatly reduced rookie salaries, which changed teams approach for allocating cap space across all positions. This change made the running back market more transparent and comparable across the seasons, making the 2011 season the best starting point for this analysis.

In [ ]:
#Preview rb_top_5 table
print("=== RB TOP 5 ===")
rb_df = pd.read_sql("SELECT * FROM rb_top_5 LIMIT 10", conn)
print(format_df(rb_df).to_string())

# Preview team_data table
print("\n=== TEAM DATA ===")
team_df = pd.read_sql("SELECT * FROM team_data LIMIT 10", conn)
print(team_df.to_string())

## Analysis
Now we join both tables to examine how teams with a top-paid running back actually performed each season. Each row shows a top-paid running back and their team's season outcome, this allows us to directly compare salary investment at the running back position and the post season result.

In [ ]:
#Join both tables to see RB salary vs team outcome
query = '''
    SELECT 
        r.player_name,
        r.team,
        r.season,
        r.contract_average,
        r.cap_hit,
        r.cap_percentage,
        t.wins,
        t.losses,
        t.made_playoffs,
        t.playoff_result
    FROM rb_top_5 r
    JOIN team_data t ON r.team = t.team AND r.season = t.season
    ORDER BY r.season DESC
'''
df = pd.read_sql(query, conn)
format_df(df)

## Key Findings

### Finding 1: Most teams with a top-paid RB missed the playoffs
Nearly 2 out of every 3 teams that committed significant money to a running back failed to even make the playoffs that season.

In [ ]:
#Playoff outcome distribution
query = '''
    SELECT 
        playoff_result as outcome,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM team_data), 2) as percentage
    FROM team_data
    GROUP BY outcome
    ORDER BY count DESC
'''
df_outcomes = pd.read_sql(query, conn)
df_outcomes['percentage'] = df_outcomes['percentage'].apply(lambda x: '{}%'.format(x))
df_outcomes

In [ ]:
#Chart 1: Playoff Outcome Distribution
img = plt.imread('charts/chart1_outcome_distribution.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

### Finding 2: Only 1 out of 105 team seasons resulted in a Super Bowl win
The 2024 Philadelphia Eagles with Saquon Barkley are the only team in this dataset to win the Super Bowl while having a top-paid running back since 2011.

In [ ]:
#Super Bowl wins by top paid RB teams
query = '''
    SELECT 
        COUNT(*) as total_team_seasons,
        SUM(CASE WHEN playoff_result = 'Super Bowl Win' THEN 1 ELSE 0 END) as super_bowl_wins,
        ROUND(SUM(CASE WHEN playoff_result = 'Super Bowl Win' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_percentage
    FROM team_data
'''
df_sb = pd.read_sql(query, conn)
df_sb['win_percentage'] = df_sb['win_percentage'].apply(lambda x: '{}%'.format(x))
df_sb

In [ ]:
#Chart 2: Average Wins by Playoff Result
img = plt.imread('charts/chart2_avg_wins.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

### Finding 3: Paying more at RB does not correlate with more wins
Teams that missed the playoffs had a higher average cap hit at RB than teams that made the Super Bowl. Investing heavily in a running back did not guarantee success but also shows the team's front office not having a good understanding on how a modern Super Bowl champion's roster, let alone playoff roster should look like.

It is important to note that Saquon Barkley only qualified by contract average and not cap hit, meaning the Eagles were not taking a significant cap burden from him that season, which allowed them to structure their salary cap efficiently.

In [ ]:
# Average cap hit by playoff result
query = '''
    SELECT 
        t.playoff_result as outcome,
        ROUND(AVG(r.cap_hit), 2) as avg_cap_hit,
        ROUND(AVG(r.cap_percentage), 2) as avg_cap_percentage
    FROM rb_top_5 r
    JOIN team_data t ON r.team = t.team AND r.season = t.season
    GROUP BY outcome
    ORDER BY avg_cap_hit DESC
'''
df_cap = pd.read_sql(query, conn)
df_cap['avg_cap_hit'] = df_cap['avg_cap_hit'].apply(lambda x: '${:,.0f}'.format(x) if pd.notnull(x) else 'N/A')
df_cap['avg_cap_percentage'] = df_cap['avg_cap_percentage'].apply(lambda x: '{}%'.format(x) if pd.notnull(x) else 'N/A')
df_cap

In [ ]:
#Chart 3: Cap Hit vs Wins Scatter Plot
img = plt.imread('charts/chart3_caphit_vs_wins.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

### Finding 4: The trend has not improved over time
Even as NFL teams have become more analytics driven, the pattern remains consistent with the underperformane of teams come playoff time. There was no meaningful or significant improvement when it came to a team with a high paying running back.

In [ ]:
#Chart 4: Pie Chart Outcome Breakdown
img = plt.imread('charts/chart4_pie_outcomes.png')
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

#Chart 5: Cap Percentage Over Time
img = plt.imread('charts/chart5_cap_pct_over_time.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

### Finding 5: Super Bowl winning teams paid their RB nearly half of what top paid RB teams committed

When comparing the average RB cap hit of every Super Bowl winning team since 2011 against the top paid RB teams in this dataset, the gap is significant. Super Bowl winning teams paid their running back an average of $3,256,230 (2.43% of cap), while top paid RB teams committed an average of $6,868,709 (4.36% of cap). This shows that championship rosters are built by spending efficiently at running back, not by paying a premium for the position. Meaning teams that invest heavily in a premium running back have not realized yet that paying for a top running back should be the last piece to a championship roster, not the first one.

In [ ]:
#Super Bowl winning teams RB cap vs top paid RB teams cap
query = '''
    SELECT 
        'Super Bowl Winners' as group_name,
        ROUND(AVG(cap_hit), 2) as avg_cap_hit,
        ROUND(AVG(cap_percentage), 2) as avg_cap_percentage
    FROM sb_winners_rb
    UNION ALL
    SELECT 
        'Top Paid RB Teams' as group_name,
        ROUND(AVG(cap_hit), 2) as avg_cap_hit,
        ROUND(AVG(cap_percentage), 2) as avg_cap_percentage
    FROM rb_top_5
'''
df_comparison = pd.read_sql(query, conn)
df_comparison['avg_cap_hit'] = df_comparison['avg_cap_hit'].apply(lambda x: '${:,.0f}'.format(x))
df_comparison['avg_cap_percentage'] = df_comparison['avg_cap_percentage'].apply(lambda x: '{}%'.format(x))
df_comparison

In [ ]:
#Chart 6: Super Bowl Winners vs Top Paid RB Teams Cap Hit Comparison
categories = ['Super Bowl Winners', 'Top Paid RB Teams']
avg_cap_hits = [3256230.60, 6868709.22]
colors = ['#27ae60', '#d9534f']

plt.figure(figsize=(8, 6))
bars = plt.bar(categories, avg_cap_hits, color=colors, width=0.4)
plt.title('Average RB Cap Hit: Super Bowl Winners vs Top Paid RB Teams (2011-2025)', fontsize=13, fontweight='bold')
plt.ylabel('Average Cap Hit ($)')
plt.ylim(0, 9000000)
for bar, val in zip(bars, avg_cap_hits):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100000, '${:,.0f}'.format(val), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/chart6_sb_vs_toppaid.png', dpi=150)
plt.show()

## Conclusion

The data tells a clear and consistent story that 

**Key takeaways:**
- **63.81%** of teams with a top-paid RB missed the playoffs entirely
- Only **0.95%** of team seasons resulted in a Super Bowl win
- Teams that missed the playoffs had a higher average RB cap hit than Super Bowl teams
- The one exception - the 2024 Philadelphia Eagles - only qualified by contract average, not cap hit
- Super Bowl winning teams paid their RB an average of **$3,256,230 (2.43% of cap)** — nearly half of what top paid RB teams committed at **$6,868,709 (4.36% of cap)**

This analysis supports the idea that it is not wise to invest heavy on a running back, but rather invest in other more important postions on the roster that can create a greater impact on the field and towards team success. Obviously it is not the running back's fault for why the team is bad, there is two sides of the ball in the sport and a running back can only do so much in a game. This shows that teams that are heavily invested in a running back do not have the pieces already set for the team itself to success on the field. The teams in the dataset that did not make the playoffs suffer from horrible front offices and management that still believe building a offense around a running back is the key to success. The era of a "running back by commitee" (RBBC) is more common nowadays in the modern NFL. 

Before Saquon Barkley signed with the Philadelphia Eagles, in the previous season the 2023 Eagles finished 11-6 and lost in the wild card to the Tampa Bay Buccaneers. But at one point in the 2023 regular season the Eagles were 10-1 and looked like the most dominant team in the league, the point that I am trying to make it the 2024 Eagles already had the foundations of a championship team. The addition of Barkley was essentially icing on the cake to a team that already had the roaster to be dominant.

In [ ]:
#Close database connection
conn.close()